# Step 2: Build MedDRA PT Lookup from MEDDRA.xlsx

**File structure (4 useful sheets, graph/ontology format):**

| Sheet | Rows | Columns | Purpose |
|-------|------|---------|--------|
| `_ID2NAME` | 85,392 | Number, Name | Maps numeric ID → term name |
| `_ID2HIERARCHY` | 85,392 | ID, HIERARCHY | Maps ID → level (LLT/PT/HLT/HGLT/SOC) |
| `_BROADER` | 121,606 | ID, BROADER ID | Parent-child relationships (child → parent) |
| `_DICTIONARY` | 155,276 | name, curie | LLT terms with codes (backup) |

**Key insight:** Every PT directly lists its SOC as a parent in `_BROADER`.
No deep traversal needed — one hop from PT → find SOC among its parents.

**Output:** `data/meddra_lookup.json` — `{pt_name_lower: {meddra_pt, meddra_pt_code, meddra_llt, meddra_llt_code, meddra_soc, meddra_soc_code}}`

In [22]:
%pip install pandas openpyxl rapidfuzz tqdm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import pandas as pd
import json
from pathlib import Path
from tqdm.notebook import tqdm

MEDDRA_FILE = Path("MEDDRA.xlsx")
DATA_DIR    = Path("data")
DATA_DIR.mkdir(exist_ok=True)

assert MEDDRA_FILE.exists(), f"MEDDRA.xlsx not found at {MEDDRA_FILE.resolve()}"
print(f"File: {MEDDRA_FILE.resolve()}  ({MEDDRA_FILE.stat().st_size / 1e6:.1f} MB)")

File: D:\Projects - AI ML LLMs\TCS-Hackathon\MEDDRA.xlsx  (27.3 MB)


In [24]:
# ── Load the 3 structural sheets ──────────────────────────────────────────────
print("Loading sheets...")
xl = pd.ExcelFile(MEDDRA_FILE, engine="openpyxl")

df_names  = xl.parse("_ID2NAME")        # Number (int) | TYPE OF DATA | Name (str)
df_levels = xl.parse("_ID2HIERARCHY")   # ID (int)     | TYPE OF DATA | HIERARCHY (LLT/PT/HLT/HGLT/SOC)
df_broad  = xl.parse("_BROADER")        # ID (int)     | TYPE OF DATA | BROADER ID (int)

print(f"  _ID2NAME:      {len(df_names):,} rows")
print(f"  _ID2HIERARCHY: {len(df_levels):,} rows")
print(f"  _BROADER:      {len(df_broad):,} rows")
print()

# Level distribution
print("Term counts by hierarchy level:")
print(df_levels["HIERARCHY"].value_counts().to_string())

Loading sheets...
  _ID2NAME:      85,392 rows
  _ID2HIERARCHY: 85,392 rows
  _BROADER:      121,606 rows

Term counts by hierarchy level:
HIERARCHY
LLT     58471
PT      24820
HLT      1737
HGLT      337
SOC        27


In [25]:
# ── Build core dictionaries ───────────────────────────────────────────────────
print("Building lookup dictionaries...")

# ID → name  (e.g. 10000002 → '11-beta-hydroxylase deficiency')
id_to_name = dict(zip(df_names["Number"], df_names["Name"]))

# ID → level  (e.g. 10000002 → 'PT')
id_to_level = dict(zip(df_levels["ID"], df_levels["HIERARCHY"]))

# child_id → [parent_id, ...]  (each term can have multiple parents = multi-axial mapping)
child_to_parents = {}
for _, row in df_broad.iterrows():
    child_to_parents.setdefault(int(row["ID"]), []).append(int(row["BROADER ID"]))

# Reverse: parent_id → [child_id, ...]  (needed to find LLTs for each PT)
parent_to_children = {}
for child, parents in child_to_parents.items():
    for p in parents:
        parent_to_children.setdefault(p, []).append(child)

# Group IDs by level
level_ids = {}
for id_, level in id_to_level.items():
    level_ids.setdefault(level, []).append(id_)

print(f"  id_to_name:        {len(id_to_name):,} entries")
print(f"  id_to_level:       {len(id_to_level):,} entries")
print(f"  child_to_parents:  {len(child_to_parents):,} entries")
print(f"  PT IDs:            {len(level_ids.get('PT', [])):,}")
print(f"  LLT IDs:           {len(level_ids.get('LLT', [])):,}")
print(f"  SOC IDs:           {len(level_ids.get('SOC', [])):,}")

Building lookup dictionaries...
  id_to_name:        85,392 entries
  id_to_level:       85,392 entries
  child_to_parents:  85,365 entries
  PT IDs:            24,820
  LLT IDs:           58,471
  SOC IDs:           27


In [17]:
# ── Verify that PTs directly link to SOC ─────────────────────────────────────
# Key finding: each PT has its SOC listed as one of its direct parents in _BROADER.
# This makes SOC lookup trivial — no recursive traversal needed.

print("Verifying PT → SOC direct links...")
pt_ids = level_ids.get("PT", [])
sample = pt_ids[:5]

for pt_id in sample:
    parents = child_to_parents.get(pt_id, [])
    parent_info = [(p, id_to_level.get(p, "?"), id_to_name.get(p, "?")) for p in parents]
    pt_name = id_to_name.get(pt_id, "?")
    soc = next((name for pid, lvl, name in parent_info if lvl == "SOC"), "NOT FOUND")
    print(f"  PT: {pt_name!r:<45} → SOC: {soc!r}")

# Coverage check
pts_with_soc = sum(
    1 for pt_id in pt_ids
    if any(id_to_level.get(p) == "SOC" for p in child_to_parents.get(pt_id, []))
)
print(f"\nPTs with direct SOC link: {pts_with_soc:,} / {len(pt_ids):,} ({pts_with_soc/len(pt_ids):.1%})")

Verifying PT → SOC direct links...
  PT: '11-beta-hydroxylase deficiency'              → SOC: 'Congenital, familial and genetic disorders'
  PT: '17 ketosteroids urine'                       → SOC: 'Investigations'
  PT: '17 ketosteroids urine decreased'             → SOC: 'Investigations'
  PT: '17 ketosteroids urine increased'             → SOC: 'Investigations'
  PT: '17 ketosteroids urine normal'                → SOC: 'Investigations'

PTs with direct SOC link: 24,820 / 24,820 (100.0%)


In [ ]:
# ── Build PT lookup ───────────────────────────────────────────────────────────
# LLT = PT for all entries.
# Rationale: FAERS reports at the PT level — the verbatim term is unknown.
# In MedDRA, when the verbatim matches a PT exactly, LLT = PT is correct practice.
# Using the "first LLT child" was arbitrary and produced truncated/unrepresentative names.

print("Building PT lookup...")

pt_lookup = {}
skipped = 0

for pt_id in tqdm(pt_ids, desc="Building lookup"):
    pt_name = id_to_name.get(pt_id, "").strip()
    if not pt_name:
        skipped += 1
        continue

    # SOC: find the parent with level == SOC (direct link, one hop)
    parents  = child_to_parents.get(pt_id, [])
    soc_id   = next((p for p in parents if id_to_level.get(p) == "SOC"), None)
    soc_name = id_to_name.get(soc_id, "") if soc_id else ""

    # LLT = PT (correct when verbatim term is unknown)
    pt_lookup[pt_name.lower()] = {
        "meddra_pt":       pt_name,
        "meddra_pt_code":  pt_id,
        "meddra_llt":      pt_name,   # LLT = PT
        "meddra_llt_code": pt_id,     # same code
        "meddra_soc":      soc_name,
        "meddra_soc_code": soc_id,
    }

print(f"Built {len(pt_lookup):,} PT entries (skipped {skipped} with no name)")

# Verify with the entries that had problems before
check_pts = ["myocardial infarction", "anaphylactic reaction", "nausea", "headache"]
print()
for key in check_pts:
    e = pt_lookup.get(key, {})
    print(f"  {e.get('meddra_pt','?'):<35}  LLT: {e.get('meddra_llt','?'):<35}  SOC: {e.get('meddra_soc','?')}")

In [27]:
# ── Fuzzy matching ────────────────────────────────────────────────────────────
from rapidfuzz import process, fuzz

ALL_PT_KEYS = list(pt_lookup.keys())

def lookup_pt(pt_name: str, threshold: int = 80) -> dict | None:
    """Look up a PT name. Exact match first, then fuzzy."""
    key = pt_name.lower().strip()
    if key in pt_lookup:
        return pt_lookup[key]
    result = process.extractOne(key, ALL_PT_KEYS, scorer=fuzz.WRatio, score_cutoff=threshold)
    if result:
        return pt_lookup[result[0]]
    return None


# Test with common MedDRA terms
test_terms = [
    "Myocardial infarction", "Nausea", "Bronchospasm",
    "Anaphylactic reaction", "Drug-induced liver injury",
    "Dyspnoea", "Hepatic failure", "Fluid retention",
    "Gastrointestinal haemorrhage", "Headache",   # from your FAERS sample
]

print(f"{'PT Name':<35} {'Matched PT':<35} {'SOC'}")
print("-" * 95)
for term in test_terms:
    r = lookup_pt(term)
    if r:
        print(f"{term:<35} {r['meddra_pt']:<35} {r['meddra_soc']}")
    else:
        print(f"{term:<35} NOT FOUND")

PT Name                             Matched PT                          SOC
-----------------------------------------------------------------------------------------------
Myocardial infarction               Myocardial infarction               Cardiac disorders
Nausea                              Nausea                              Gastrointestinal disorders
Bronchospasm                        Bronchospasm                        Respiratory, thoracic and mediastinal disorders
Anaphylactic reaction               Anaphylactic reaction               Immune system disorders
Drug-induced liver injury           Drug-induced liver injury           Hepatobiliary disorders
Dyspnoea                            Dyspnoea                            Respiratory, thoracic and mediastinal disorders
Hepatic failure                     Hepatic failure                     Hepatobiliary disorders
Fluid retention                     Fluid retention                     Metabolism and nutrition disorders
Gast

In [28]:
# ── Test against actual FAERS PT names ───────────────────────────────────────
faers_file = DATA_DIR / "faers_raw.json"
if faers_file.exists():
    with open(faers_file) as f:
        faers_records = json.load(f)

    faers_pts = list(set(r["reaction_pt"] for r in faers_records))
    print(f"Unique PT names in FAERS data: {len(faers_pts)}")

    found     = [pt for pt in faers_pts if lookup_pt(pt) is not None]
    not_found = [pt for pt in faers_pts if lookup_pt(pt) is None]

    print(f"Matched:   {len(found)} ({len(found)/len(faers_pts):.1%})")
    print(f"Unmatched: {len(not_found)} ({len(not_found)/len(faers_pts):.1%})")

    if not_found:
        print(f"\nUnmatched PT names (first 15):")
        for pt in not_found[:15]:
            print(f"  {pt!r}")
else:
    print("FAERS data not found. Run notebook 01 first.")

Unique PT names in FAERS data: 1722
Matched:   1719 (99.8%)
Unmatched: 3 (0.2%)

Unmatched PT names (first 15):
  'Pseudocroup'
  'Grand mal convulsion'
  'Blighted ovum'


In [29]:
# ── Save lookup ───────────────────────────────────────────────────────────────
OUTPUT_LOOKUP = DATA_DIR / "meddra_lookup.json"
with open(OUTPUT_LOOKUP, "w") as f:
    json.dump(pt_lookup, f)

print(f"Saved {len(pt_lookup):,} entries to {OUTPUT_LOOKUP}")
print(f"File size: {OUTPUT_LOOKUP.stat().st_size / 1e6:.1f} MB")

Saved 24,820 entries to data\meddra_lookup.json
File size: 6.1 MB


## Done

MedDRA lookup saved to `data/meddra_lookup.json`.

**Before running notebook 03**, start the vLLM server in a terminal:
```bash
vllm serve Qwen/Qwen3-32B --dtype bfloat16 --max-model-len 4096 --port 8000
```
Wait for `INFO: Application startup complete.` then run notebook 03.